In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ishanshrivastava28/tata-online-retail-dataset")
print("Path to dataset files:", path)

file_path = path + '/Online Retail Data Set.csv'

import pandas as pd
import numpy as np

# Confirm Date and Time is correct
data = pd.read_csv(file_path, encoding='latin1')
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], dayfirst=True)

#
# Filter Quantity and UnitPrice to avoid no value
data = data.dropna(subset=['CustomerID'])
data = data[(data['Quantity'] > 0) & (data['UnitPrice'] > 0)]
data['CustomerID'] = data['CustomerID'].astype(int)

data['amount'] = data['UnitPrice'] * data['Quantity']
data = data[['CustomerID', 'InvoiceDate', 'amount', 'Country']]

# Cohort by Month

data['event_month'] = data['InvoiceDate'].dt.to_period('M')

data['cohort_month'] = (
    data.groupby('CustomerID')['event_month']
    .transform('min')
)

data['cohort_index'] = (
    (data['event_month'].dt.year - data['cohort_month'].dt.year) * 12
    + (data['event_month'].dt.month - data['cohort_month'].dt.month)
)

# Cohort Counts

cohort_counts = (
    data.groupby(['cohort_month', 'cohort_index'])['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'active_users'})
)

cohort_table = cohort_counts.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='active_users'
)

base = cohort_table[0]
cohort_table = cohort_table.divide(base, axis=0)

retention = cohort_table.round(3) * 100
retention.index = retention.index.astype(str)

retention = retention.fillna(0.0)
print(retention)
# Revenue Retention

cohort_revenue = (
    data.groupby(['cohort_month', 'cohort_index'])['amount']
    .sum()
    .reset_index()
)

cohort_revenue_table = cohort_revenue.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='amount'
)
rev_base = cohort_revenue_table[0].replace(0, np.nan)

revenue_retention = (
    cohort_revenue_table
    .divide(rev_base, axis=0)
    .round(3)*100
)
revenue_retention = revenue_retention.fillna(0.0)
revenue_retention.index = revenue_retention.index.astype(str)


# Cohort Avg Revenue per Retained User

cohort_average = pd.merge(cohort_counts, cohort_revenue, on=['cohort_month', 'cohort_index'])
cohort_average['Avg Revenue'] = (cohort_average['amount'] / cohort_average['active_users']).round(2)
cohort_average_table = cohort_average.pivot_table(
    index='cohort_month',
    columns='cohort_index',
    values='Avg Revenue'
).fillna(0.0)

# Cohort by Week

data['event_week'] = data['InvoiceDate'].dt.to_period('W').apply(lambda r: r.start_time)
data['cohort_week'] = data.groupby('CustomerID')['event_week'].transform('min')

data['cohort_week_index'] = (
    (data['event_week'] - data['cohort_week']).dt.days // 7
)

weekly_counts = (
    data.groupby(['cohort_week','cohort_week_index'])['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID':'active_users'})
)

## Cohort table for Weekly Value

weekly_pivot = weekly_counts.pivot_table(
    index='cohort_week', columns='cohort_week_index', values='active_users'
)

weekly_retention = (weekly_pivot.divide(weekly_pivot[0], axis=0) * 100).fillna(0.0)

# Country Segmentation

## Count unique users by country, cohort_index
country_counts = (
    data
    .groupby(['Country', 'cohort_index'])['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'users'})
)

# Separate month 0 and month 1
month0 = country_counts[country_counts['cohort_index'] == 0][['Country', 'users']] \
    .rename(columns={'users': 'month0_users'})

month1 = country_counts[country_counts['cohort_index'] == 1][['Country', 'users']] \
    .rename(columns={'users': 'month1_users'})

month1_retention = (
    month0
    .merge(month1, on='Country', how='left')
)

month1_retention['month1_retention_pct'] = (
    month1_retention['month1_users'] /
    month1_retention['month0_users'] * 100
)

month1_retention = month1_retention.fillna(0.0)

top_countries = (
    month1_retention
    .sort_values('month0_users', ascending=False)
    .head(5)
)

top_countries = top_countries.sort_values(
    'month1_retention_pct', ascending=False
)


# Country Segmentation Bar Chart
import matplotlib.pyplot as plt
plt.figure()
plt.bar(
    top_countries['Country'],
    top_countries['month1_retention_pct']
)

plt.xlabel('Country')
plt.ylabel('Month-1 Retention (%)')
plt.title('Month-1 Customer Retention by Country')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Customer Retention Heatmap
fig, ax = plt.subplots(figsize=(12, 6))

data_for_heatmap = retention.values
im = ax.imshow(data_for_heatmap, aspect='auto', cmap='YlGnBu')

ax.set_xlabel('Cohort Index (Months)')
ax.set_ylabel('Cohort Month')

ax.set_xticks(np.arange(len(retention.columns)))
ax.set_yticks(np.arange(len(retention.index)))

ax.set_xticklabels(retention.columns)
ax.set_yticklabels(retention.index)

plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

for i in range(len(retention.index)):
    for j in range(len(retention.columns)):
        ax.text(j, i, f'{data_for_heatmap[i, j]:.1f}',
                ha='center', va='center', color='black', fontsize=8)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Retention (%)')

plt.title('Customer Retention Cohort Heatmap')
plt.tight_layout()
plt.show()

# Revenue Retention Heatmap

fig, ax = plt.subplots(figsize=(12, 6))

data_for_heatmap = revenue_retention.values
im = ax.imshow(data_for_heatmap, aspect='auto', cmap='Oranges')

ax.set_xlabel('Cohort Index (Months)')
ax.set_ylabel('Cohort Month')

ax.set_xticks(np.arange(len(revenue_retention.columns)))
ax.set_yticks(np.arange(len(revenue_retention.index)))

ax.set_xticklabels(revenue_retention.columns)
ax.set_yticklabels(revenue_retention.index)

plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

for i in range(len(revenue_retention.index)):
    for j in range(len(revenue_retention.columns)):
        ax.text(j, i, f'{data_for_heatmap[i, j]:.1f}',
                ha='center', va='center', color='black', fontsize=8)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Revenue Retention (%)')

plt.title('Revenue Retention Cohort Heatmap')
plt.tight_layout()
plt.show()

# Avg Revenue per Retained User Heatmap

fig, ax = plt.subplots(figsize=(12, 6))

data_for_heatmap = cohort_average_table.values
im = ax.imshow(data_for_heatmap, aspect='auto', cmap='Oranges')

ax.set_xlabel('Cohort Index (Months)')
ax.set_ylabel('Cohort Month')

ax.set_xticks(np.arange(len(cohort_average_table.columns)))
ax.set_yticks(np.arange(len(cohort_average_table.index)))

ax.set_xticklabels(cohort_average_table.columns)
ax.set_yticklabels(cohort_average_table.index)

plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

for i in range(len(cohort_average_table.index)):
    for j in range(len(cohort_average_table.columns)):
        ax.text(j, i, f'{data_for_heatmap[i, j]:.1f}',
                ha='center', va='center', color='black', fontsize=8)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Avg Revenue per Retained User ($)')

plt.title('Avg Revenue per Retained User Cohort Heatmap')
plt.tight_layout()
plt.show()